In [23]:
# --- system setup ---
import sys
import os
sys.path.append(os.path.abspath(".."))

In [24]:
from ib_insync import Stock, util
from ibkr.Class_IBKR_IB import IBKR_IB
ibkr = IBKR_IB(port=7496)

async def start_ibkr():
    await ibkr.connect()
    print("IBKR connected:", ibkr.ib.isConnected())

await start_ibkr()

IBKR connected: True


Error 321, reqId 4: Error validating request.-'bH' : cause - Historical data bar size setting is invalid. Legal ones are: 1 secs, 5 secs, 10 secs, 15 secs, 30 secs, 1 min, 2 mins, 3 mins, 4 mins, 5 mins, 10 mins, 15 mins, 20 mins, 30 mins, 1 hour, 2 hours, 3 hours, 4 hours, 8 hours, 1 day, 1W, 1M, contract: Stock(conId=677037663, symbol='ARKB', exchange='SMART', primaryExchange='BATS', currency='USD', localSymbol='ARKB', tradingClass='ARKB')
Peer closed connection.


In [28]:
lookback_period = "30 D"
length_of_each_period = "15 mins"
use_regular_trading_hours = True

prices_to_use = "MIDPOINT"  #TRADES FOR BTC FUTURES

'''

    "TRADES"
    "MIDPOINT"
    "BID"
    "ASK"
    "BID_ASK"
    "ADJUSTED_LAST"
    "HISTORICAL_VOLATILITY"
    "OPTION_IMPLIED_VOLATILITY"
    "FEE_RATE"
    "REBATE_RATE"
    "SCHEDULE"
        
'''

'\n\n    "TRADES"\n    "MIDPOINT"\n    "BID"\n    "ASK"\n    "BID_ASK"\n    "ADJUSTED_LAST"\n    "HISTORICAL_VOLATILITY"\n    "OPTION_IMPLIED_VOLATILITY"\n    "FEE_RATE"\n    "REBATE_RATE"\n    "SCHEDULE"\n\n'

In [29]:
symbol_list = ['ARKB',
                'BITB',
                'BRRR',
                'BTC',
                'BTCW',
                'EZBC',
                'FBTC',
                'GBTC',
                'HODL',
                'IBIT']


In [30]:
dict_ = {}

for sym in symbol_list:

    contract = Stock(sym, 'SMART', 'USD')
    await ibkr.ib.qualifyContractsAsync(contract)

    bars = await ibkr.ib.reqHistoricalDataAsync(
                contract=contract,
                endDateTime="",
                durationStr=lookback_period,
                barSizeSetting=length_of_each_period,
                whatToShow=prices_to_use,
                useRTH=use_regular_trading_hours,
                formatDate=1,
                keepUpToDate=False
            )

    df = util.df(bars)
    df['symbol'] = sym
    dict_[sym] = df

In [31]:
import pandas as pd

dfs = [df for df in dict_.values()]

df_all = pd.concat(dfs, ignore_index=True)

df_all.drop(columns=['open', 'high', 'low', 'volume', 'average', 'barCount'], inplace=True)

df_all['time'] = df_all['date'].dt.time
df_all['date'] = df_all['date'].dt.date

df_all

,date,close,symbol,time
0,2026-04-28,25.30,ARKB,09:30:00
1,2026-04-28,25.29,ARKB,09:45:00
2,2026-04-28,25.23,ARKB,10:00:00
3,2026-04-28,25.16,ARKB,10:15:00
4,2026-04-28,25.16,ARKB,10:30:00
...,...,...,...,...
7645,2026-06-09,34.90,IBIT,11:00:00
7646,2026-06-09,34.70,IBIT,11:15:00
7647,2026-06-09,34.71,IBIT,11:30:00
7648,2026-06-09,34.59,IBIT,11:45:00


In [42]:
# --- utils ---
from input_output.Standard_Output   import standard_output
from input_output.Class_InputOutput import InputOutput
io = InputOutput()

wb, ws = io.set_xw_book_and_sheet('2026 BTC ETF Ratios.xlsx', "BTC RATIOS")

scalar_df = io.get_xw_df(ws, "btc_ratios", table=True)

scalar_df['Date'] = pd.to_datetime(scalar_df['Date']).dt.date

scalar_df

,DOW,Date,ARKB,BITB,BRRR,BTC,BTCW,EZBC,FBTC,GBTC,HODL,IBIT
0,Sunday,2026-03-01,3012.9766,1840.9591,3543.8313,2260.2814,945.3954,1729.1018,1147.9526,1283.0366,3535.8029,1763.6799
1,Monday,2026-03-02,3012.9939,1840.9692,3543.8556,2260.2907,945.4019,1729.1108,1147.9605,1283.0893,3535.8029,1763.6920
2,Tuesday,2026-03-03,3013.0112,1840.9793,3543.8799,2260.3000,945.4084,1729.1198,1147.9684,1283.1420,3535.8029,1763.7041
3,Wednesday,2026-03-04,3013.0285,1840.9894,3543.9042,2260.3093,945.4149,1729.1288,1147.9763,1283.1947,3535.8029,1763.7162
4,Thursday,2026-03-05,3013.0458,1840.9995,3543.9285,2260.3186,945.4214,1729.1378,1147.9842,1283.2474,3535.8029,1763.7283
...,...,...,...,...,...,...,...,...,...,...,...,...
360,Wednesday,2027-02-24,3019.2254,1844.5951,3552.5793,2263.6294,947.7354,1732.3418,1150.7966,1302.1593,3535.8029,1768.0359
361,Thursday,2027-02-25,3019.2428,1844.6052,3552.6036,2263.6387,947.7419,1732.3508,1150.8045,1302.2128,3535.8029,1768.0480
362,Friday,2027-02-26,3019.2602,1844.6153,3552.6279,2263.6480,947.7484,1732.3598,1150.8124,1302.2663,3535.8029,1768.0601
363,Saturday,2027-02-27,3019.2776,1844.6254,3552.6522,2263.6573,947.7549,1732.3688,1150.8203,1302.3198,3535.8029,1768.0722


In [45]:
for indx in df_all.index:
    sym = df_all.loc[indx, 'symbol']
    date = df_all.loc[indx, 'date']

    scalar = scalar_df.loc[scalar_df['Date'] == date, sym]

    df_all.iloc[indx, df_all.columns.get_loc('scalar') ] = scalar

df_all['unit_value'] = df_all['close'] * df_all['scalar']

df_all

,date,close,symbol,time,scalar,unit_value
0,2026-04-28,25.30,ARKB,09:30:00,3013.9800,76253.694000
1,2026-04-28,25.29,ARKB,09:45:00,3013.9800,76223.554200
2,2026-04-28,25.23,ARKB,10:00:00,3013.9800,76042.715400
3,2026-04-28,25.16,ARKB,10:15:00,3013.9800,75831.736800
4,2026-04-28,25.16,ARKB,10:30:00,3013.9800,75831.736800
...,...,...,...,...,...,...
7645,2026-06-09,34.90,IBIT,11:00:00,1764.8899,61594.657510
7646,2026-06-09,34.70,IBIT,11:15:00,1764.8899,61241.679530
7647,2026-06-09,34.71,IBIT,11:30:00,1764.8899,61259.328429
7648,2026-06-09,34.59,IBIT,11:45:00,1764.8899,61047.541641


In [50]:
group_cols = ["date", "time"]

idx_high = df_all.groupby(group_cols)["unit_value"].idxmax()
idx_low  = df_all.groupby(group_cols)["unit_value"].idxmin()

df_high = df_all.loc[idx_high, group_cols + ["symbol", "unit_value"]].rename(
    columns={"symbol": "symbol_high", "unit_value": "unit_high"}
)

df_low = df_all.loc[idx_low, group_cols + ["symbol", "unit_value"]].rename(
    columns={"symbol": "symbol_low", "unit_value": "unit_low"}
)

df_high_low = df_high.merge(df_low, on=group_cols)

df_high_low

,date,time,symbol_high,unit_high,symbol_low,unit_low
0,2026-04-28,09:30:00,BRRR,76293.579864,HODL,76231.910524
1,2026-04-28,09:45:00,ARKB,76223.554200,IBIT,76168.357989
2,2026-04-28,10:00:00,HODL,76090.478408,EZBC,76034.262248
3,2026-04-28,10:15:00,HODL,75842.972205,EZBC,75774.818678
4,2026-04-28,10:30:00,BITB,75853.234431,BRRR,75797.246166
...,...,...,...,...,...,...
760,2026-06-09,11:00:00,BRRR,61634.021394,GBTC,61581.686440
761,2026-06-09,11:15:00,EZBC,61259.363738,FBTC,61239.468006
762,2026-06-09,11:30:00,HODL,61310.822286,BRRR,61243.932651
763,2026-06-09,11:45:00,BRRR,61102.082199,GBTC,61040.592124


In [51]:
counts = (
    pd.concat(
        [
            df_high_low["symbol_high"].value_counts().rename("high_count"),
            df_high_low["symbol_low"].value_counts().rename("low_count")
        ],
        axis=1
    )
    .fillna(0)
    .astype(int)
    .reset_index()
    .rename(columns={"index": "symbol"})
)

counts

,symbol,high_count,low_count
0,BTCW,283,10
1,HODL,138,80
2,BRRR,131,113
3,BTC,102,15
4,ARKB,56,66
5,BITB,28,18
6,IBIT,21,147
7,GBTC,3,49
8,EZBC,3,208
9,FBTC,0,59
